# Memory Management & Performance — Expert Interview Guide

Covers: reference counting, cyclic GC, `sys.getsizeof`, `tracemalloc`, profiling, `__slots__`, `memoryview`.

> **Key model:** CPython uses reference counting + cyclic GC. Objects are freed the instant refcount hits 0 (no GC needed).

## Key Topics

1. **Reference Counting** - Every Python object has a reference count. When it drops to 0, the object is immediately freed. `sys.getrefcount(x)` always returns count+1 (the function call adds a reference).

2. **Cyclic Garbage Collector** - Reference counting can't handle cycles (`A -> B -> A`). CPython has a secondary cyclic GC with 3 generations.

3. **`sys.getsizeof` — Object Sizes** - `getsizeof` is **shallow** — it doesn't count referenced objects.

4. **`tracemalloc` — Finding Memory Leaks** - Common leak sources: (1) unbounded caches/dicts, (2) closures capturing large objects, (3) global lists accumulating data, (4) event handlers keeping references to dead objects.

5. **Profiling — `timeit` and `cProfile`** - Profile before optimizing — 80% of time is in 20% of code. `cProfile` finds hot functions; `line_profiler` finds the specific hot line. Never guess.

6. **Memory Optimization — `array`, `memoryview`, `deepcopy`**

   - `array` module — compact typed arrays vs list

   - `memoryview` — zero-copy buffer slicing (critical for binary I/O)

   - Shallow vs deep copy

## Reference Counting

In [ ]:
import sys

x = [1, 2, 3]
print(f'After creation: {sys.getrefcount(x)}')

y = x  # second reference
print(f'After y=x:      {sys.getrefcount(x)}')

del y
print(f'After del y:    {sys.getrefcount(x)}')

**Interview Insight:** CPython's refcounting is **deterministic** — objects free instantly when refcount=0. This is why `with open(...) as f:` guarantees file close. Java/Go GC is non-deterministic.

## Cyclic Garbage Collector

Reference counting can't handle cycles (`A -> B -> A`). CPython has a secondary cyclic GC with 3 generations.

In [ ]:
import gc

class Node:
    def __init__(self, name): self.name = name; self.next = None
    def __del__(self): print(f'  Freed: {self.name}')

# Create a cycle — refcounting alone can't free these
a = Node('A'); b = Node('B')
a.next = b; b.next = a  # A -> B -> A

print('Deleting references...')
del a; del b
print('Not freed yet! (cycle keeps refcount > 0)')

collected = gc.collect()  # trigger cyclic GC
print(f'GC freed: {collected} objects')

**Interview Insight:** The cyclic GC runs when generation-0 object count exceeds threshold (default 700). Disable it during allocation-heavy loops for performance: `gc.disable()` ... `gc.enable(); gc.collect()`.

## Profiling Example

In [ ]:
import timeit, cProfile

# timeit: micro-benchmarks
setup = 'd = {i: i for i in range(10000)}; lst = list(range(10000))'
dict_t = timeit.timeit('5000 in d', setup=setup, number=100000)
list_t = timeit.timeit('5000 in lst', setup=setup, number=100000)
print(f'Dict lookup: {dict_t:.4f}s  O(1)')
print(f'List search: {list_t:.4f}s  O(n)')

**Interview Insight:** `memoryview` is critical for high-performance binary I/O — slicing a `bytes` object creates a copy (O(n)), slicing a `memoryview` creates a view (O(1)). Used internally by many network/file protocols.

See 05_memory_performance.ipynb for full implementation with working examples and demonstrations.